# ????? ??? ???

###  "?? ??? : ??? ????? ?? ??"

### ??
    * ??? ISBN?? ??? ??? ???
    * ??? : ??? 1??
    * topics : 5
    * ??? : 5

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf

%matplotlib inline

import matplotlib.pyplot as plt
import re
import urllib.request
from konlpy.tag import Okt


from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

import urllib

In [ ]:
import konlpy
from konlpy.tag import Okt

In [ ]:
mirrored_strategy = tf.distribute.MirroredStrategy()

# ??? ???

## ?? ?? ??

In [ ]:
import glob
import pandas as pd
from pandas.core.dtypes.missing import notnull
import nltk
from konlpy.tag import Okt
from wordcloud import WordCloud, STOPWORDS
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import platform
from matplotlib import font_manager, rc
import re
from gensim.models import Word2Vec
from pandas import DataFrame
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

## <???>

In [ ]:
def cleansing(text):
    # text = str(text)
    pattern = "([a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+.[a-zA-Z0-9-.]+)"  # e-mail ?? ??
    text = re.sub(pattern=pattern, repl=" ", string=text)

    pattern = "(http|ftp|https)://(?:[-\w.]|(?:\da-fA-F]{2}))+"  # url ??
    text = re.sub(pattern=pattern, repl=" ", string=text)

    pattern = "([^??????-?a-zA-Z0-9])+"  # ?? ??, ?? ?? - without '?,?,?,?,?'
    text = re.sub(pattern=pattern, repl=" ", string=text)

    pattern = "<[^>]*>"  # html tag ??
    text = re.sub(pattern=pattern, repl=" ", string=text)

    pattern = "[\r|\n]"  # \r, \n ??
    text = re.sub(pattern=pattern, repl=" ", string=text)

    pattern = re.compile(r"\s+")  # ?? space ??
    text = re.sub(pattern=pattern, repl=" ", string=text)

    return text

### ISBN? ???? ??? ?? ? ???

## <???>

#### ?? ??? 

In [ ]:
okt = Okt()

----

### ?? ??? ????

In [ ]:
# ????
import pickle

book_morphs_with_isbn_pos_non_stopword = []
with open('data/detail_bestbook_unit_isbn_with_isbn_pos_non_stopword.txt', 'rb') as lf:
    book_morphs_with_isbn_pos_non_stopword = pickle.load(lf)
    
print(book_morphs_with_isbn_pos_non_stopword[0])

In [ ]:
print(len(book_morphs_with_isbn_pos_non_stopword))

print(book_morphs_with_isbn_pos_non_stopword[10])

In [ ]:
print(book_morphs_with_isbn_pos_non_stopword[0])

In [ ]:
#  ?? ???? ??
stopwords = ['?','?','?','?','?','?','?','?','?','?','?','?','??','?','?','?','?','??','?','??','??','??','?','??',
             '??','??','?','?','??','??','??','?','???','???','?','??','???','???','?','??','?','?','?','?','?','?','?',
             '??','??','??','???','??','??','?','?','???','?','?','?','?','?','?','?','?','??','?','???','?','?','???','??',
            '?','??','??','?','?','??','?','?','?','??','??','?','??','?','?','?','?','?','?','?','?','?','?','?','?', '??', '?', '?'
            , '??','??','?','??','??','???']

In [ ]:
######### ??? ??? ??? ?? + ???(1??) + ??? ??

book_train_nouns_adj = []
book_train_isbn = []
for (isbn, test) in book_morphs_with_isbn_pos_non_stopword :
    tts = [(word,pos) for (word,pos) in test if not word in stopwords] #??? ??

    na = [word for (word,pos) in tts if pos in 'Noun'] # ??? ??
    for (word,pos) in tts : #??? : ??? 1??
        if pos == 'Adjective' and len(word)>1 :
            na.append(word)
    if na : 
        book_train_nouns_adj.append(na)
        book_train_isbn.append(isbn)

print('**********************END***************************')
print(len(book_train_nouns_adj))
print(len(book_train_isbn))

In [ ]:
for i in range(0,5) :
#     print(book_morphs_with_isbn_pos_non_stopword[i])
    print(book_train_nouns_adj[i])

----

# ?? ???? ?? ?? ???

(????, ???)

#### ??? ?
* ??? ?? > 1

In [ ]:
from gensim import corpora


dictionary = corpora.Dictionary(book_train_nouns_adj)
corpus = [dictionary.doc2bow(text) for text in book_train_nouns_adj]

In [ ]:
print(len(book_train_nouns_adj))
print(len(dictionary))
print(len(corpus))

In [ ]:
print(corpus[24983])

## ?? ?? & ?? ???

In [ ]:
import matplotlib.pyplot as plt
from gensim.models import CoherenceModel
from gensim.models.ldamodel import LdaModel 
from gensim.models.callbacks import CoherenceMetric 
from gensim import corpora 
from gensim.models.callbacks import PerplexityMetric 

coherence_values=[]

for i in range(2,20) : 
    ldamodel = LdaModel(corpus,num_topics=i,id2word=dictionary)
    coherence_model_lda = CoherenceModel(model=ldamodel,texts=book_train_nouns_adj,dictionary=dictionary) # ,topn=10
    coherence_lda = coherence_model_lda.get_coherence()
    coherence_values.append(coherence_lda)

In [ ]:
print('coherence_values : ',coherence_values)

In [ ]:
x=range(2,20,1)
plt.plot(x,coherence_values)
plt.xlabel('number of topics')
plt.ylabel('coherence score')
plt.show()

In [ ]:
perplexity_values=[]

for i in range(2,20) :
    ldamodel = LdaModel(corpus,num_topics=i,id2word=dictionary)
    perplexity_values.append(ldamodel.log_perplexity(corpus))

In [ ]:
x=range(2,20)
plt.plot(x,perplexity_values)
plt.xlabel('number of topics')
plt.ylabel('perplexity score')
plt.show()

In [ ]:
print('perplexity_values : ',perplexity_values)

#### (???? ?? : ?? ??? ??? ? coherence? ?? ?? perplexity? ??? ?? 15?? ??? ? ??)

----

### Topics = 15 , Passes ???

In [ ]:
coherence_values_passes=[]

for i in range(5,20) : 
    ldamodel = LdaModel(corpus,num_topics=15,id2word=dictionary,passes=i)
    coherence_model_lda = CoherenceModel(model=ldamodel,texts=book_train_nouns_adj,dictionary=dictionary) # ,topn=10
    coherence_lda = coherence_model_lda.get_coherence()
    coherence_values_passes.append(coherence_lda)

In [ ]:
print('coherence_values_passes : ',coherence_values_passes)

In [ ]:
x=range(5,20)
plt.plot(x,coherence_values_passes)
plt.xlabel('number of passes')
plt.ylabel('coherence score')
plt.show()

#### (???? ?? : passes? 6? ????? ?? ? ??..)

----

### ?? ?? ? ???

In [ ]:
ldamodel = LdaModel(corpus,
                    num_topics=15,
                    id2word=dictionary,
                    passes=6) # ??? ??? ??.... alpha
topics = ldamodel.print_topics(num_words=8)
for topic in topics:
    print(topic)

In [ ]:
import pyLDAvis.gensim

pyLDAvis.enable_notebook()
vis = pyLDAvis.gensim.prepare(ldamodel, corpus, dictionary)
pyLDAvis.display(vis)

In [ ]:
for i, topic_list in enumerate(ldamodel[corpus]):
    if i==20:
        break
#     print(i,'?? ??? topic ???',topic_list)
    print(book_train_isbn[i],'? topic ???',topic_list)

In [ ]:
# ?? ??
model_name = 'model/nouns_adj_15topics_6passes'
ldamodel.save(model_name)

9788955820072 ? topic ??? [(2, 0.3888356), (4, 0.32910487), (12, 0.2153306)]  
<???? ?? ??? ??>  
2 - ??  
4 - ???  
12 - ???  
  
---  
  
???? > ??? > ??/??/??? > ??? ??  
???? > ??? > ??1~2?? > ??/??/??  
???? > ?? > ??? > ??/?? ???  
???? > ?? > ??? > ????/???  
???? > ?? > ?? ??/?? > ?? ????/?? > ??   
???? > ?? > ?? ??/?? > ?? ?? ??  
???? > ???? > ??/???? ???? > ??? ??? ??? ????  

---

? ?? ?? ? ??? '???'??. ?? ??? ??? ???? ??? ?? ??? ??? ?? ???? ?????, ? ?? ??? ?? ??? ?? ??? ???? ????? ?? ??. ??? ?? ??, ????? ??? ????, ??? ?? ??? ????.

?? ??? ??? ???? ??? ??? ???? ?, ??, ??, ??? ?? ??? ????. ????, ??????, ??????, ??, ????, ???, ???... ???? ?? ??? ?? ????? ?? ??? ??? ??? ?????. ??? ???? ????? ?? ??? ???.

??? ?? ?? ?? ?? ? ?? ????, ??? ???? ?? ? ????. ???? ????, ?????? ???? ??? ????? ?? ??? ??? ? ???. ? ?? ??? ??? ???? ??? ?? ???? ???? ?? ? ???? ?? ?? ????.

??? ? ?? ?? ??? ???? ?? ??? ??? ??? ???? ???, ? ? ???? ??? ???? ???? ?? ????, ??? ?? ???? ?? ???. ??? ??? ? ?? ??? ??? ????, ? ???? ??? ?? ?? ??? ???? ??.

??? ???? ?? ??? ???? ???? ?? 5??? ??? ???? 2??? ???? ???? ?? ???? ??? ?????. ?? ????? ?? ???? ??? ? ??? ???? ?? ???? ???? ???? ??? ???? ?, ??? ??? ??? ??? ??.

In [ ]:
# (0, '0.035*"??" + 0.017*"??" + 0.010*"??" + 0.009*"??" + 0.008*"??" + 0.008*"???" + 0.008*"??" + 0.007*"??"')
# (1, '0.019*"??" + 0.017*"??" + 0.017*"??" + 0.014*"??" + 0.013*"??" + 0.010*"??" + 0.010*"???" + 0.009*"??"')
# (2, '0.046*"??" + 0.020*"??" + 0.019*"???" + 0.017*"????" + 0.015*"??" + 0.012*"????" + 0.012*"?" + 0.011*"?"')
# (3, '0.028*"??" + 0.020*"??" + 0.018*"?" + 0.012*"??" + 0.012*"??" + 0.012*"??" + 0.011*"???" + 0.010*"?"')
# (4, '0.020*"?" + 0.018*"??" + 0.016*"??" + 0.015*"??" + 0.013*"???" + 0.012*"?" + 0.012*"??" + 0.007*"??"')
# (5, '0.040*"?" + 0.033*"??" + 0.024*"??" + 0.021*"??" + 0.018*"??" + 0.018*"?" + 0.017*"??" + 0.015*"??"')
# (6, '0.019*"??" + 0.019*"??" + 0.018*"???" + 0.016*"????" + 0.016*"??" + 0.015*"???" + 0.012*"??" + 0.012*"??"')
# (7, '0.048*"??" + 0.026*"??" + 0.020*"??" + 0.016*"??" + 0.013*"??" + 0.012*"??" + 0.011*"??" + 0.011*"??"')
# (8, '0.054*"??" + 0.049*"??" + 0.039*"???" + 0.031*"??" + 0.031*"??" + 0.029*"??" + 0.020*"??" + 0.019*"?"')
# (9, '0.022*"??" + 0.018*"???" + 0.013*"??" + 0.012*"??" + 0.011*"?" + 0.011*"??" + 0.009*"??" + 0.009*"??"')
# (10, '0.107*"??" + 0.028*"????" + 0.022*"????" + 0.021*"??" + 0.018*"??" + 0.013*"??" + 0.013*"??" + 0.012*"??"')
# (11, '0.036*"??" + 0.027*"??" + 0.023*"??" + 0.018*"??" + 0.017*"??" + 0.016*"???" + 0.013*"??" + 0.013*"??"')
# (12, '0.020*"??" + 0.020*"??" + 0.019*"??" + 0.019*"??" + 0.013*"??" + 0.011*"??" + 0.010*"??" + 0.009*"??"')
# (13, '0.046*"??" + 0.024*"??" + 0.015*"??" + 0.015*"???" + 0.015*"????" + 0.013*"????" + 0.010*"???" + 0.010*"??"')
# (14, '0.112*"??" + 0.060*"??" + 0.028*"??" + 0.021*"?" + 0.017*"??" + 0.015*"??" + 0.014*"??" + 0.012*"??"')

---



9788974825515 ? topic ??? [(7, 0.7403959), (10, 0.100506864), (14, 0.08626847)]  
<? ??? ??? ???? ????? YCT 1?>  
  
7 - ??   
10 - ??? ???   
14 - ??, ??    

---  
???? > ??? > ??? ???  

---

???? ??? ?? ????? ???? ???? ???? ??? ? ?? ???? ?. ? ???? ???? ???? ??? ?? ????, ???? ???? ??? ??? ?? ?? ??? ??? ??? ???. ????? 3? ????, ???? ? ??? ??? ??? ??? ??? ??, ??, ?? ??? ???? ????.

